# SignalShap - Review Round 8, remaining runs

Everything the round-8 review still needs, in cost order. All the code and
tests are already on `main`; this notebook only *runs* it on the real corpora.

**Read this first.** Do not use `SignalShap_M4_FullStudy.ipynb` for these.
That notebook runs `yelp2018`, `gowalla` and `amazon_book_lgcn`, which are the
untimestamped LightGCN splits and **not** the three corpora this paper
reports. The paper's corpora are `ml_1m`, `amazon_video_games` and
`gowalla_ts`. Running the M4 notebook will produce artefacts, and they will be
the wrong ones.

| Stage | Review item | Cost | Artefact |
|---|---|---|---|
| 1 | 3 - blocked retirement | minutes | `global_timeblock.json` |
| 2 | 10 - neutral pools, ML-1M | minutes | `pool_sensitivity.json` |
| 3 | 7 - ten-seed refreshed history | ~1 h | `protocol_sensitivity.json` |
| 4 | 10 - neutral pools, Amazon | ~10 min | `pool_sensitivity.json` |
| 5-7 | 6 - regenerate legacy diagnostics | hours | `results_*.json` |
| 8 | 10 - neutral pools, Gowalla | longest | `pool_sensitivity.json` |
| 9 | manifest refresh | seconds | `MANIFEST.json` |

Every stage is independent and resumable. If one fails, the rest still run and
the ones that succeeded stay on disk.

If you would rather not babysit a notebook, the identical sequence is one
shell command:

```bash
bash scripts/run_round8_remaining.sh 24
```

## 1 - Environment

No `%pip install` here: use the environment the repository declares, so the artefacts record the versions the paper was produced with.

In [1]:
import os, sys, platform, subprocess, time, json
from pathlib import Path

def _find_repo():
    """Locate the repo by its CONTENTS, not by a guessed path.

    An earlier version hardcoded `Path.home()/"signalshap-code"` and checked
    only `.exists()`. A stale empty directory at that path passed the check,
    the notebook chdir'd into it, and all nine stages died instantly with
    "can't open file scripts/run_study.py" while the real clone sat elsewhere.
    A directory existing is not evidence that it is the right directory, so
    probe for files that must be present.
    """
    MARKERS = ("scripts/run_study.py", "src/signalshap/config.py",
               "configs/frozen.yaml")

    def looks_right(p):
        return p and all((p / m).exists() for m in MARKERS)

    # 1. the notebook's own location, walking up; correct no matter where the
    #    repo lives, and the only candidate that needs no configuration.
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if looks_right(cand):
            return cand
    # 2. a few conventional spots, for a notebook opened from elsewhere.
    for cand in (Path.home() / "signalshap-code",
                 Path.home() / "Desktop" / "signalshap-code"):
        if looks_right(cand):
            return cand
    return None


REPO = _find_repo()
if REPO is None:
    raise SystemExit(
        "Could not locate the signalshap-code repo.\n"
        "Set it explicitly and re-run this cell:\n"
        "    REPO = Path('/full/path/to/signalshap-code')\n"
        "    os.chdir(REPO); sys.path.insert(0, str(REPO/'src'))\n"
        f"(searched from {Path.cwd()} upward, and the usual home locations)")

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

# The interpreter running the stages must be THIS kernel's interpreter. The
# notebook shells out with sys.executable; if the kernel is a bare system
# python while the repo has a .venv, the stages get a different environment
# than the cells. Warn rather than guess.
_venv = REPO / ".venv" / "bin" / "python"
if _venv.exists() and Path(sys.executable).resolve() != _venv.resolve():
    print(f"WARNING: kernel is {sys.executable}")
    print(f"         repo venv is {_venv}")
    print("         Stages will use the KERNEL. Select the .venv kernel if "
          "that is not what you want.\n")

# A missing corpus must be a hard error, never a silent synthetic fallback.
os.environ["SIGNALSHAP_STRICT_DATA"] = "1"

print("python :", platform.python_version(), "|", platform.machine())
try:
    ram = int(subprocess.check_output(["sysctl", "-n", "hw.memsize"]).strip()) / 1e9
    print(f"RAM    : {ram:.0f} GB")
except Exception:
    ram = None
    print("RAM    : unknown (not macOS)")

import numpy as np, pandas as pd, scipy, sklearn
print("numpy", np.__version__, "| pandas", pd.__version__,
      "| scipy", scipy.__version__, "| sklearn", sklearn.__version__)
print("cwd    :", Path.cwd())

python : 3.12.13 | arm64
RAM    : 52 GB
numpy 2.5.1 | pandas 3.0.5 | scipy 1.18.0 | sklearn 1.9.0
cwd    : /Users/mlouhichi/Desktop/personal/phd/next-paper/signalshap-code


### The one knob

`BUDGET_GB` caps how much RAM the five dense score matrices may use. The user
count per corpus is *derived* from it, not guessed, so a corpus downsizes
instead of being OOM-killed.

**Leave it at 24 unless you have a reason.** The reported corpus shapes
(6,038 x 3,533 / 7,120 x 3,516 / 8,865 x 82,134) were produced at 24 GB, and
`check_paper_shape` will refuse to overwrite the paper's artefacts if a
different budget yields a different shape. That guard is the point: a
different budget silently means a different Gowalla.

In [2]:
BUDGET_GB = 24
CORPORA = ["ml_1m", "amazon_video_games", "gowalla_ts"]

from signalshap.memory import PAPER_CORPUS_SHAPE, default_budget_gb, free_gb
print(f"budget      : {default_budget_gb(BUDGET_GB):.1f} GB "
      f"(free: {free_gb():.1f} GB)")
print("paper shapes:")
for k, (u, i) in PAPER_CORPUS_SHAPE.items():
    print(f"  {k:20s} {u:6,} users x {i:7,} items")

budget      : 24.0 GB (free: 14.5 GB)
paper shapes:
  ml_1m                 6,038 users x   3,533 items
  amazon_video_games    7,120 users x   3,516 items
  gowalla_ts            8,865 users x  82,134 items


### Corpora

`ml-1m` is fetched from GroupLens rather than vendored: its README forbids
redistribution. Gowalla check-ins come from SNAP. Both scripts are idempotent,
so re-running costs nothing if the data is already there.

In [3]:
need = [p for p in ["data/raw/ml-1m/ratings.dat",
                    "data/raw/gowalla_ts",
                    "data/raw/amazon_video_games"]
        if not Path(p).exists()]
print("missing:", need or "nothing, all corpora present")

if need:
    for script in ("scripts/fetch_benchmarks.sh", "scripts/fetch_timestamped.sh"):
        print(f"\n$ bash {script}")
        r = subprocess.run(["bash", script], capture_output=True, text=True)
        print((r.stdout or "")[-2000:])
        if r.returncode != 0:
            print("STDERR:", (r.stderr or "")[-1000:])

missing: nothing, all corpora present


### A helper that does not hide failures

Long stages fail for boring reasons: a missing corpus, a full disk, a typo.
This runner streams output live, records the exit status, and *keeps going*,
so one bad stage does not discard hours of successful work in the stages after
it. That is the same reason `run_round8_remaining.sh` does not use `set -e`.

In [4]:
RESULTS = {}

def run(label, *args, allow_fail=True):
    """Run one stage, stream its output, remember whether it worked."""
    cmd = [sys.executable, *args]
    print("=" * 72)
    print(label)
    print("  $ " + " ".join(str(c) for c in cmd))
    print("=" * 72, flush=True)
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    dt = time.time() - t0
    ok = p.returncode == 0
    RESULTS[label] = {"ok": ok, "seconds": round(dt, 1), "returncode": p.returncode}
    print(f"\n--- {'ok' if ok else 'FAILED'} ({label}) in {dt:.0f}s\n", flush=True)
    if not ok and not allow_fail:
        raise RuntimeError(label)
    return ok

## 2 - Stage 1: blocked retirement (review item 3)

The main split is leave-last-out *within* each user, which is standard but not
globally causal: up to 44% of pooled training events postdate the median test
event. The paper's central operational claim is about **retirement**, so that
claim in particular needs checking under a split where nothing is fitted on
the future.

Expect coverage to collapse (only ~8.7% of users survive a global cutoff) and
the blocked corpus to fail the paper's own recall gate at 0.464. That is a
known and disclosed property, not a bug: read this as a stress test on a
different, smaller population, not as a cleaner measurement of the same one.

In [5]:
run("item 3: blocked retirement (ml_1m)",
    "scripts/run_global_timeblock.py", "--corpora", "ml_1m",
    "--budget-gb", "24", "--retirement")

item 3: blocked retirement (ml_1m)
  $ /Users/mlouhichi/Desktop/personal/phd/next-paper/signalshap-code/.venv/bin/python scripts/run_global_timeblock.py --corpora ml_1m --budget-gb 24 --retirement
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
    [retirement] removing cf...
    [retirement] removing ct...
    [retirement] removing pop...
    [retirement] removing rec...
    [retirement] removing seq...
ml_1m: 526/6,038 users retained, recall 0.464, order cf > seq > pop > ct > rec, flips ['pop', 'seq']
   retirement: tau_LOO=+0.80 tau_Shapley=+0.60 (advantage +0.20); cheapest truly pop, LOO says pop, Shapley says rec

--- ok (item 3: blocked retirement (ml_1m)) in 56s



True

In [6]:
from signalshap.config import read_artefact

tb = read_artefact("global_timeblock.json").get("ml_1m", {})
print(f"users retained : {tb.get('users_retained'):,} of {tb.get('users_original'):,} "
      f"({tb.get('user_coverage', 0):.1%})")
print(f"candidate recall: {tb.get('candidate_recall', 0):.3f}  (gate 0.60)")
print(f"train events after any test event: {tb.get('train_events_after_any_test_event')}"
      "   <- must be 0, that is the whole point")
print(f"ordering        : {' > '.join(tb.get('ordering', []))}")
print(f"material flips  : {tb.get('material_flips') or 'none'}")

r = tb.get("retirement")
if r:
    print("\nRETIREMENT UNDER THE BLOCKED SPLIT")
    print(f"  tau LOO vs truth     : {r['kendall_tau_loo_vs_truth']:+.2f}")
    print(f"  tau Shapley vs truth : {r['kendall_tau_shapley_vs_truth']:+.2f}")
    print(f"  LOO advantage        : {r['tau_advantage_loo_minus_shapley']:+.2f}")
    print(f"  cheapest truly       : {r['cheapest_to_retire_true']}")
    print(f"  LOO picks            : {r['cheapest_by_loo']} "
          f"({'correct' if r['loo_picks_correctly'] else 'WRONG'})")
    print(f"  Shapley picks        : {r['cheapest_by_shapley']} "
          f"({'correct' if r['shapley_picks_correctly'] else 'WRONG'})")
    print("\nIf the LOO advantage is <= 0 here, that WEAKENS the paper's claim")
    print("and must be reported as such, not buried. Tell me either way.")
else:
    print("\nno retirement block: was --retirement passed?")

users retained : 526 of 6,038 (8.7%)
candidate recall: 0.464  (gate 0.60)
train events after any test event: 0   <- must be 0, that is the whole point
ordering        : cf > seq > pop > ct > rec
material flips  : ['pop', 'seq']

RETIREMENT UNDER THE BLOCKED SPLIT
  tau LOO vs truth     : +0.80
  tau Shapley vs truth : +0.60
  LOO advantage        : +0.20
  cheapest truly       : pop
  LOO picks            : pop (correct)
  Shapley picks        : rec (WRONG)

If the LOO advantage is <= 0 here, that WEAKENS the paper's claim
and must be reported as such, not buried. Tell me either way.


## 3 - Stage 2: neutral candidate pools (review item 10)

The candidate pool is coalition-independent, which is what the game requires,
but it is not *source*-independent: the five players build it, so it is
enriched for the items they rank highly. These runs rebuild the whole game on
pools that consult no source score at all.

Three rules. `popularity` is neutral for four players but correlated with
`pop`, which we state rather than gloss. `random` is strictly neutral, but
recall collapses to roughly N/|I|, so most users lose a retrievable target and
the game flattens; a null result there would be a null *game*, not pool
robustness. `random_oracle` restores the target while keeping the distractors
source-blind, and it is the informative one.

**Compare orderings and signs, not levels.** Each rule gives a different
`|C_u|`, hence a different baseline and recall ceiling.

In [7]:
run("item 10: neutral pools (ml_1m)",
    "scripts/run_pool_sensitivity.py", "--corpora", "ml_1m",
    "--budget-gb", "24")

item 10: neutral pools (ml_1m)
  $ /Users/mlouhichi/Desktop/personal/phd/next-paper/signalshap-code/.venv/bin/python scripts/run_pool_sensitivity.py --corpora ml_1m --budget-gb 24
== ml_1m
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
    [pool] union_of_top_n...
    [pool] popularity...
    [pool] random...
    [pool] random_oracle...
   popularity     tau=0.8 top_same=True signs=False recall=0.625
   random         tau=0.8 top_same=True signs=True recall=0.178
   random_oracle  tau=0.8 top_same=True signs=False recall=1.000

--- ok (item 10: neutral pools (ml_1m)) in 62s



True

In [8]:
def show_pools(corpus):
    # read_artefact raises if the file is absent, which is the normal state
    # before the stage above has ever succeeded. Report that, do not traceback.
    p = Path("artefacts/pool_sensitivity.json")
    if not p.exists():
        print(f"{corpus}: pool_sensitivity.json not written yet"); return
    blob = json.loads(p.read_text()).get(corpus)
    if not blob:
        print(f"{corpus}: not run yet"); return
    rows = []
    for label, p in blob["pools"].items():
        ag = blob["agreement"].get(label, {})
        rows.append({
            "pool": label,
            "recall": round(p["candidate_recall"], 3),
            "v(G)": round(p["v_grand"], 5),
            "ordering": " > ".join(p["ordering"]),
            "flips": ",".join(p["material_flips"]) or "-",
            "tau vs union": (None if label == "union_of_top_n"
                             else round(ag.get("kendall_tau_vs_union") or 0, 2)),
            "same top": ("-" if label == "union_of_top_n"
                         else ag.get("same_top_source")),
        })
    display(pd.DataFrame(rows))

show_pools("ml_1m")

,pool,recall,v(G),ordering,flips,tau vs union,same top
0,union_of_top_n,0.748,0.05169,seq > cf > pop > rec > ct,cf,NaN,-
1,popularity,0.625,0.05898,seq > cf > pop > ct > rec,-,0.8,True
2,random,0.178,0.03064,seq > cf > pop > ct > rec,cf,0.8,True
3,random_oracle,1.000,0.17754,seq > cf > pop > ct > rec,"cf,pop",0.8,True


## 4 - Stage 3: ten-seed refreshed history (review item 7)

The frozen-versus-refreshed protocol contrast was seed 42 only, which cannot
separate a protocol effect from the game's own run-to-run noise. This runs the
paired contrast across the same ten seeds as the main results and reports a
percentile-bootstrap interval plus a per-source sign count.

This is the slow one on this page: ten seeds, two full games each. Roughly an
hour on MovieLens.

In [9]:
run("item 7: ten-seed refreshed history (ml_1m)",
    "scripts/run_protocol_sensitivity.py", "--corpora", "ml_1m",
    "--budget-gb", "24",
    "--seeds", *[str(s) for s in range(42, 52)])

item 7: ten-seed refreshed history (ml_1m)
  $ /Users/mlouhichi/Desktop/personal/phd/next-paper/signalshap-code/.venv/bin/python scripts/run_protocol_sensitivity.py --corpora ml_1m --budget-gb 24 --seeds 42 43 44 45 46 47 48 49 50 51
budget 24.0 GB | corpora ['ml_1m']
  resuming; already have ['amazon_video_games', 'gowalla_ts', 'ml_1m']
== ml_1m: temporal state
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
   tau=1.00 maxdelta=1.46e-02 top seq->seq
== ml_1m: temporal state across 10 seeds
    [temporal] seed 42...
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
    [temporal] seed 43...
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
    [temporal] seed 44...
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
    [temporal] seed 45...
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
    [tempora

True

In [10]:
ps = read_artefact("protocol_sensitivity.json").get("ml_1m", {}).get("temporal_seeds")
if not ps:
    print("not run yet (needs --seeds)")
else:
    rc = ps["relative_change_v_grand"]
    print(f"seeds                  : {ps['n_seeds']}")
    print(f"v(G) frozen            : {ps['v_grand_frozen']['mean']:.5f} "
          f"(sd {ps['v_grand_frozen']['sd']:.5f})")
    print(f"v(G) refreshed         : {ps['v_grand_refreshed']['mean']:.5f} "
          f"(sd {ps['v_grand_refreshed']['sd']:.5f})")
    print(f"relative change        : {rc['mean']:+.1%} "
          f"[{rc['ci']['lo']:+.1%}, {rc['ci']['hi']:+.1%}]  "
          f"({rc['n_positive']}/{ps['n_seeds']} seeds positive)")
    print(f"Kendall tau mean       : {ps['kendall_tau']['mean']:.3f}  "
          f"({ps['kendall_tau']['n_unit']}/{ps['n_seeds']} seeds preserve the ordering)")
    print(f"top source changes on  : {ps['n_seeds_top_source_changes']}/{ps['n_seeds']} seeds")
    display(pd.DataFrame([
        {"source": g,
         "mean delta": round(d["mean"], 6),
         "ci lo": round(d["ci"]["lo"], 6),
         "ci hi": round(d["ci"]["hi"], 6),
         "+/-": f"{d['n_positive']}/{d['n_negative']}",
         "sign stable": d["sign_stable"]}
        for g, d in ps["per_source_delta"].items()]))

seeds                  : 10
v(G) frozen            : 0.05225 (sd 0.00088)
v(G) refreshed         : 0.06880 (sd 0.00086)
relative change        : +31.7% [+30.2%, +33.2%]  (10/10 seeds positive)
Kendall tau mean       : 1.000  (10/10 seeds preserve the ordering)
top source changes on  : 0/10 seeds


,source,mean delta,ci lo,ci hi,+/-,sign stable
0,cf,0.001988,0.001554,0.002470,10/0,True
1,ct,0.000001,-0.000108,0.000103,5/5,False
2,pop,0.000565,0.000492,0.000650,10/0,True
3,rec,0.000059,-0.000006,0.000133,5/5,False
4,seq,0.013930,0.013687,0.014184,10/0,True


## 5 - Stage 4: neutral pools on Amazon-VG

In [11]:
run("item 10: neutral pools (amazon_video_games)",
    "scripts/run_pool_sensitivity.py", "--corpora", "amazon_video_games",
    "--budget-gb", "24")
show_pools("amazon_video_games")

item 10: neutral pools (amazon_video_games)
  $ /Users/mlouhichi/Desktop/personal/phd/next-paper/signalshap-code/.venv/bin/python scripts/run_pool_sensitivity.py --corpora amazon_video_games --budget-gb 24
  resuming; already have ['ml_1m']
amazon_video_games: ~10,242 items (est, probe saw 3,414) -> 73,227 users fit a 24 GB budget
== amazon_video_games
[signalshap] amazon_video_games: loaded REAL data (7,120 users x 3,516 items, 117,468 interactions)
    [pool] union_of_top_n...
    [pool] popularity...
    [pool] random...
    [pool] random_oracle...
   popularity     tau=0.8 top_same=True signs=True recall=0.421
   random         tau=0.8 top_same=True signs=True recall=0.171
   random_oracle  tau=0.6 top_same=False signs=True recall=0.972

--- ok (item 10: neutral pools (amazon_video_games)) in 173s



,pool,recall,v(G),ordering,flips,tau vs union,same top
0,union_of_top_n,0.589,0.04130,cf > seq > ct > pop > rec,-,NaN,-
1,popularity,0.421,0.04100,cf > seq > pop > ct > rec,ct,0.8,True
2,random,0.171,0.02056,cf > seq > pop > ct > rec,ct,0.8,True
3,random_oracle,0.972,0.11898,seq > cf > pop > ct > rec,-,0.6,False


## 6 - Stages 5-7: regenerate the legacy-rule diagnostics (review item 6)

Table 1's recall and monotonicity counts, the duplicate-injection test, the
segment profiles, the lambda and candidate-cap sweeps, the grand-pool
ablation and the estimand comparison were all produced under the *legacy*
candidate truncation key, before the source-symmetric rule became the method
of record. They are currently labelled as single-seed sensitivity analyses.
Re-running them under the final rule promotes them to confirmatory.

**One corpus per call.** `run_study.py` refuses a multi-corpus invocation when
the corpora need different user caps, because a single process-wide cap would
silently apply the smallest to all of them and substitute a different
MovieLens. That refusal is deliberate; do not work around it.

This is the expensive block. Gowalla alone can run for hours at 82,134 items.

In [12]:
for corpus in CORPORA:
    run(f"item 6: regenerate diagnostics ({corpus})",
        "scripts/run_study.py", "--datasets", corpus,
        "--budget-gb", "24", "--seeds", "42", "43", "44")

item 6: regenerate diagnostics (ml_1m)
  $ /Users/mlouhichi/Desktop/personal/phd/next-paper/signalshap-code/.venv/bin/python scripts/run_study.py --datasets ml_1m --budget-gb 24 --seeds 42 43 44
memory budget: 24.0 GB (free: 23.6 GB)
ml_1m: ~10,599 items (est, probe saw 3,533) -> 70,761 users fit a 24 GB budget
  ml_1m: loaded 6,038 users x 3,533 items (0.4 GB of scores)
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)
[signalshap] ml_1m: loaded REAL data (6,038 users x 3,533 items, 575,276 interactions)

complete in 430s
  ml_1m        recall=0.748 PASS  monotonicity 26/80

--- ok (item 6: regenerate diagnostics (ml_1m)) in 437s

item 6: regenerate diagnostics (amazon_video_games)
  $ /Users/mlouhichi/Desktop/personal/phd/next-paper/signalshap-code/.venv/bin/python scripts/run

In [13]:
for corpus in CORPORA:
    p = Path(f"artefacts/results_{corpus}.json")
    if not p.exists():
        print(f"{corpus:20s} MISSING"); continue
    e0 = json.loads(p.read_text())["e0a_candidates"]
    print(f"{corpus:20s} rule={e0.get('candidate_rule')}  "
          f"recall={e0['candidate_recall']:.3f}  "
          f"{'PASS' if e0['gate_passes'] else 'below gate (rung 2)'}")
print("\nrule should read symmetric_reciprocal_rank on all three once this finishes.")

ml_1m                rule=symmetric_reciprocal_rank  recall=0.748  PASS
amazon_video_games   rule=symmetric_reciprocal_rank  recall=0.589  below gate (rung 2)
gowalla_ts           rule=None  recall=0.450  below gate (rung 2)

rule should read symmetric_reciprocal_rank on all three once this finishes.


## 7 - Stage 8: neutral pools on Gowalla (optional, longest)

Four pool rules x 32 coalitions at `N_max = 11,623` over an 82,134-item
catalogue. Skip it if you are short on time: the ordering-invariance claim is
already supported by the two denser corpora, and Gowalla's absolute values are
diluted by half anyway (half its users have a test venue already in training,
so their payoff is identically zero in every coalition).

In [14]:
RUN_GOWALLA_POOLS = True     # set False to skip

if RUN_GOWALLA_POOLS:
    run("item 10: neutral pools (gowalla_ts)",
        "scripts/run_pool_sensitivity.py", "--corpora", "gowalla_ts",
        "--budget-gb", "24")
    show_pools("gowalla_ts")
else:
    print("skipped")

item 10: neutral pools (gowalla_ts)
  $ /Users/mlouhichi/Desktop/personal/phd/next-paper/signalshap-code/.venv/bin/python scripts/run_pool_sensitivity.py --corpora gowalla_ts --budget-gb 24
  resuming; already have ['amazon_video_games', 'ml_1m']
gowalla_ts: ~84,594 items (est, probe saw 28,198) -> 8,865 users fit a 24 GB budget
== gowalla_ts
[signalshap] gowalla_ts: loaded REAL data (8,865 users x 82,134 items, 542,970 interactions)
    [pool] union_of_top_n...
    [pool] popularity...
    [pool] random...
    [pool] random_oracle...
   popularity     tau=1.0 top_same=True signs=True recall=0.157
   random         tau=0.8 top_same=False signs=True recall=0.069
   random_oracle  tau=0.8 top_same=False signs=True recall=0.501

--- ok (item 10: neutral pools (gowalla_ts)) in 2239s



,pool,recall,v(G),ordering,flips,tau vs union,same top
0,union_of_top_n,0.450,0.01690,cf > ct > seq > pop > rec,pop,NaN,-
1,popularity,0.157,0.01263,cf > ct > seq > pop > rec,pop,1.0,True
2,random,0.069,0.00726,ct > cf > seq > pop > rec,-,0.8,False
3,random_oracle,0.501,0.06061,ct > cf > seq > pop > rec,rec,0.8,False


## 8 - Refresh the hashed manifest and verify

In [15]:
run("manifest", "scripts/make_manifest.py")

manifest
  $ /Users/mlouhichi/Desktop/personal/phd/next-paper/signalshap-code/.venv/bin/python scripts/make_manifest.py
wrote artefacts/MANIFEST.json: 54 artefacts, commit 0f059815dc97

--- ok (manifest) in 1s



True

In [16]:
for checker in ("check_paper_numbers.py", "check_discover_ai.py", "check_latex.py"):
    print("=" * 72)
    r = subprocess.run([sys.executable, f"scripts/{checker}"],
                       capture_output=True, text=True)
    print(f"{checker}: exit {r.returncode}")
    print((r.stdout or "").strip()[-1500:])
    if r.stderr.strip():
        print("  stderr:", r.stderr.strip()[-600:])

check_paper_numbers.py: exit 0
PAPER/ARTEFACT MISMATCHES:
  - fuse NDCG 0.06348 not found in prose
  - global NDCG 0.06357 not found in prose
  - C4 heterogeneity '7 of 30' not found in prose
  - monotonicity MovieLens: paper says 7/23 (material/raw), artefact gives 7/26
  stderr: [rung 2] amazon_video_games: recall 0.589 below gate, exemption declared; relative contrasts only
  [rung 2] gowalla_ts: recall 0.450 below gate, exemption declared; relative contrasts only
check_discover_ai.py: exit 0
manuscript complies with Discover AI submission guidelines
check_latex.py: exit 0
latex static checks pass


## 9 - Summary

Send me this table and the artefacts it names. If anything failed, send the
error too: a stage that fails for a boring reason is much cheaper to fix than
a stage that silently produced the wrong corpus.

In [17]:
if not RESULTS:
    # An empty frame must not read as success: it means nothing ran at all,
    # which is a different situation from every stage passing.
    print("NO STAGES RAN. Did you execute the run(...) cells above?")
else:
    summary = pd.DataFrame([
        {"stage": k, "ok": v["ok"], "minutes": round(v["seconds"] / 60, 1),
         "exit": v["returncode"]}
        for k, v in RESULTS.items()])
    display(summary)

    failed = summary[~summary["ok"]]["stage"].tolist()
    print("ALL STAGES OK" if not failed else f"{len(failed)} FAILED: {failed}")
    print(f"total: {summary['minutes'].sum():.0f} min")

print("\nartefacts touched by this notebook:")
for name in ("global_timeblock.json", "pool_sensitivity.json",
             "protocol_sensitivity.json", "results_ml_1m.json",
             "results_amazon_video_games.json", "results_gowalla_ts.json",
             "MANIFEST.json"):
    p = Path("artefacts") / name
    print(f"  {'OK  ' if p.exists() else 'MISS'} {name}"
          f"{'' if not p.exists() else f'  ({p.stat().st_size / 1024:.0f} KB)'}")

,stage,ok,minutes,exit
0,item 3: blocked retirement (ml_1m),True,0.9,0
1,item 10: neutral pools (ml_1m),True,1.0,0
2,item 7: ten-seed refreshed history (ml_1m),True,16.3,0
3,item 10: neutral pools (amazon_video_games),True,2.9,0
4,item 6: regenerate diagnostics (ml_1m),True,7.3,0
5,item 6: regenerate diagnostics (amazon_video_g...,True,9.9,0
6,item 6: regenerate diagnostics (gowalla_ts),False,115.8,-9
7,item 10: neutral pools (gowalla_ts),True,37.3,0
8,manifest,True,0.0,0


1 FAILED: ['item 6: regenerate diagnostics (gowalla_ts)']
total: 191 min

artefacts touched by this notebook:
  OK   global_timeblock.json  (4 KB)
  OK   pool_sensitivity.json  (16 KB)
  OK   protocol_sensitivity.json  (26 KB)
  OK   results_ml_1m.json  (35 KB)
  OK   results_amazon_video_games.json  (34 KB)
  OK   results_gowalla_ts.json  (33 KB)
  OK   MANIFEST.json  (8 KB)


### Committing

```bash
git add artefacts/ && git commit -m "round 8: runs from the M4" \
  && git push origin arena/019fc2ce-signalshap-code
```

Then tell me it is pushed and I will fold the numbers into the manuscript,
including any that come out against the paper's framing.